In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
import os
import urllib.request
import zipfile
from collections import Counter

In [ ]:
# Download and extract 2024 survey (workaround for Cash issue with comprehension variables)
import urllib.request
import zipfile
import os

_data_dir = os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'stackoverflow')
_yr = 2024
_zip_path = os.path.join(_data_dir, 'stack-overflow-developer-survey-2024.zip')
_yr_dir = os.path.join(_data_dir, '2024')

if not os.path.exists(_zip_path):
    _url = 'https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-2024.zip'
    print('Downloading 2024 survey...')
    _opener = urllib.request.build_opener()
    _opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Cash-Benchmark/1.0; research project)')]
    urllib.request.install_opener(_opener)
    urllib.request.urlretrieve(_url, _zip_path)
    print('  Downloaded: ' + str(round(os.path.getsize(_zip_path)/1e6, 1)) + ' MB')
else:
    print('Already downloaded: ' + str(round(os.path.getsize(_zip_path)/1e6, 1)) + ' MB')

os.makedirs(_yr_dir, exist_ok=True)
with zipfile.ZipFile(_zip_path, 'r') as _zf:
    _zf.extractall(_yr_dir)
_extracted = os.listdir(_yr_dir)
print('Extracted 2024: ' + str(len(_extracted)) + ' files')
for _item in _extracted:
    _item_path = os.path.join(_yr_dir, _item)
    _item_sz = os.path.getsize(_item_path) / 1e6
    print('  ' + _item + ' (' + str(round(_item_sz, 1)) + ' MB)')

In [ ]:
# Load and combine survey data from all years
import pandas as _pd6
import os as _os6
import time as _time6

_data_dir = _os6.path.join(_os6.getcwd(), 'examples', 'large_scale_projects', 'data', 'stackoverflow')
_survey_years = [2022, 2023, 2024]
_t0 = _time6.time()

_dfs = []
for _yr in _survey_years:
    _yr_dir = _os6.path.join(_data_dir, str(_yr))
    # Find the main survey results CSV
    _csv_files = [f for f in _os6.listdir(_yr_dir) if f.endswith('.csv') and 'survey_results' in f.lower()]
    if not _csv_files:
        # Try any CSV file
        _csv_files = [f for f in _os6.listdir(_yr_dir) if f.endswith('.csv')]
    
    for _csv in _csv_files:
        _path = _os6.path.join(_yr_dir, _csv)
        _sz = _os6.path.getsize(_path) / 1e6
        if _sz < 1:  # Skip schema/metadata files
            continue
        print(f'Loading {_yr}/{_csv} ({_sz:.1f} MB)...')
        _df = _pd6.read_csv(_path, low_memory=False)
        _df['survey_year'] = _yr
        _dfs.append(_df)
        print(f'  {len(_df):,} respondents, {len(_df.columns)} columns')

_all_surveys = _pd6.concat(_dfs, ignore_index=True)
_elapsed = _time6.time() - _t0

print(f'\nCombined dataset: {len(_all_surveys):,} respondents')
print(f'Columns: {len(_all_surveys.columns)}')
print(f'Memory: {_all_surveys.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'Load time: {_elapsed:.1f}s')
print(f'\nRespondents by year:')
_year_counts = _all_surveys['survey_year'].value_counts().sort_index()
for _yr in _survey_years:
    if _yr in _year_counts.index:
        print(f'  {_yr}: {_year_counts[_yr]:,}')

In [ ]:
# ── Cell 5: Salary analysis ──
# Standardize compensation column across years
# 2022-2024 use 'ConvertedCompYearly' for USD annual compensation

# Find compensation column
_comp_cols = [c for c in _all_surveys.columns if 'comp' in c.lower() and 'yearly' in c.lower()]
if not _comp_cols:
    _comp_cols = [c for c in _all_surveys.columns if 'comp' in c.lower()]
print(f'Compensation columns found: {_comp_cols}')

_comp_col = _comp_cols[0] if _comp_cols else None
if _comp_col:
    _with_salary = _all_surveys[_all_surveys[_comp_col].notna() & (_all_surveys[_comp_col] > 0)].copy()
    # Remove extreme outliers (< $5K or > $1M)
    _with_salary = _with_salary[(_with_salary[_comp_col] >= 5000) & (_with_salary[_comp_col] <= 1000000)]
    _with_salary['salary_usd'] = _with_salary[_comp_col]
    print(f'Respondents with valid salary: {len(_with_salary):,} ({100*len(_with_salary)/len(_all_surveys):.1f}%)')
    print(f'Median salary: ${_with_salary["salary_usd"].median():,.0f}')
    print(f'Mean salary: ${_with_salary["salary_usd"].mean():,.0f}')
    print(f'Salary range: ${_with_salary["salary_usd"].min():,.0f} - ${_with_salary["salary_usd"].max():,.0f}')
    
    # Salary by year
    print(f'\nMedian salary by year:')
    _salary_by_year = _with_salary.groupby('survey_year')['salary_usd'].median()
    for _yr in _survey_years:
        if _yr in _salary_by_year.index:
            print(f'  {_yr}: ${_salary_by_year[_yr]:,.0f}')
    
    # Salary distribution buckets
    _bins = [5000, 25000, 50000, 75000, 100000, 150000, 200000, 300000, 500000, 1000000]
    _labels = ['$5-25K', '$25-50K', '$50-75K', '$75-100K', '$100-150K', '$150-200K', '$200-300K', '$300-500K', '$500K-1M']
    _with_salary['salary_bin'] = _pd6.cut(_with_salary['salary_usd'], bins=_bins, labels=_labels)
    _salary_dist = _with_salary['salary_bin'].value_counts().sort_index()
    print(f'\nSalary distribution:')
    for _idx in range(len(_salary_dist)):
        _bin = _salary_dist.index[_idx]
        _count = _salary_dist.iloc[_idx]
        _pct = 100 * _count / len(_with_salary)
        print(f'  {_bin}: {_count:,} ({_pct:.1f}%)')
else:
    print('No compensation column found!')
    _with_salary = _all_surveys.copy()
    _with_salary['salary_usd'] = 0

In [ ]:
# ── Cell 6: Technology popularity trends ──
# Analyze programming language usage across years

# Find language column (varies by year: LanguageHaveWorkedWith, etc.)
_lang_cols = [c for c in _all_surveys.columns if 'language' in c.lower() and ('worked' in c.lower() or 'have' in c.lower())]
if not _lang_cols:
    _lang_cols = [c for c in _all_surveys.columns if 'language' in c.lower()]
print(f'Language columns: {_lang_cols}')

_lang_col = _lang_cols[0] if _lang_cols else None
_lang_trends = {}

if _lang_col:
    # Languages are semicolon-separated in the survey
    for _yr in _survey_years:
        _yr_data = _all_surveys[_all_surveys['survey_year'] == _yr]
        _valid = _yr_data[_yr_data[_lang_col].notna()]
        _total = len(_valid)
        _lang_counter = Counter()
        for _val in _valid[_lang_col].values:
            if isinstance(_val, str):
                for _lang in _val.split(';'):
                    _lang_counter[_lang.strip()] += 1
        _lang_trends[_yr] = {_l: 100 * _c / _total for _l, _c in _lang_counter.most_common(20)}
        print(f'\n{_yr} - Top 10 languages (% of respondents):')
        for _rank in range(min(10, len(_lang_counter))):
            _lang_name = _lang_counter.most_common(10)[_rank][0]
            _lang_pct = 100 * _lang_counter.most_common(10)[_rank][1] / _total
            print(f'  {_rank+1:2d}. {_lang_name:20s} {_lang_pct:.1f}%')

# Show trend for key languages
_key_langs = ['JavaScript', 'Python', 'TypeScript', 'Rust', 'Go', 'Java', 'C#', 'C++', 'SQL']
print(f'\nLanguage adoption trends:')
print(f'{"Language":20s}', end='')
for _yr in _survey_years:
    print(f'  {_yr}', end='')
print()
for _lang_name in _key_langs:
    print(f'{_lang_name:20s}', end='')
    for _yr in _survey_years:
        _pct = _lang_trends.get(_yr, {}).get(_lang_name, 0)
        if _pct > 0:
            print(f'  {_pct:4.1f}%', end='')
        else:
            print(f'     -', end='')
    print()

In [ ]:
# ── Cell 7: Salary by language chart ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Calculate median salary for top languages (2024 or latest year)
_latest_yr = max(_survey_years)
_latest = _with_salary[_with_salary['survey_year'] == _latest_yr]

if _lang_col:
    _lang_salaries = {}
    _top_langs = list(_lang_trends.get(_latest_yr, {}).keys())[:15]
    for _lang_name in _top_langs:
        _mask = _latest[_lang_col].str.contains(_lang_name, na=False, regex=False)
        _lang_data = _latest[_mask]
        if len(_lang_data) >= 50:  # Need minimum sample size
            _lang_salaries[_lang_name] = _lang_data['salary_usd'].median()
    
    _lang_sal_sorted = sorted(_lang_salaries.items(), key=lambda x: x[1], reverse=True)
    _names = [x[0] for x in _lang_sal_sorted]
    _salaries = [x[1] for x in _lang_sal_sorted]
    
    _fig1, _ax1 = plt.subplots(figsize=(12, 6))
    _bars = _ax1.barh(range(len(_names)), _salaries, color='steelblue')
    _ax1.set_yticks(range(len(_names)))
    _ax1.set_yticklabels(_names)
    _ax1.set_xlabel('Median Annual Salary (USD)')
    _ax1.set_title(f'Median Developer Salary by Language ({_latest_yr})')
    _ax1.invert_yaxis()
    # Add value labels
    for _idx in range(len(_salaries)):
        _ax1.text(_salaries[_idx] + 1000, _idx, f'${_salaries[_idx]:,.0f}', va='center', fontsize=9)
    _ax1.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('examples/large_scale_projects/data/stackoverflow/salary_by_language.png', dpi=150)
    plt.show()
    
    print(f'\nSalary by language ({_latest_yr}):')
    for _name, _sal in _lang_sal_sorted:
        print(f'  {_name:20s} ${_sal:>10,.0f}')

In [ ]:
# ── Cell 8: ML salary prediction pipeline ──
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# Build features for salary prediction
# Select numeric and categorical features
_feature_candidates = ['YearsCodePro', 'YearsCode', 'EdLevel', 'OrgSize', 'Country',
                        'DevType', 'Age', 'survey_year']
_available_features = [c for c in _feature_candidates if c in _with_salary.columns]
print(f'Available features: {_available_features}')

# Prepare feature matrix
_ml_df = _with_salary[_available_features + ['salary_usd']].dropna(subset=['salary_usd']).copy()

# Convert YearsCode/YearsCodePro to numeric (they may have 'Less than 1 year', 'More than 50 years')
for _col in ['YearsCode', 'YearsCodePro']:
    if _col in _ml_df.columns:
        _ml_df[_col] = _pd6.to_numeric(_ml_df[_col], errors='coerce')

_ml_df = _ml_df.dropna()
print(f'Samples after cleanup: {len(_ml_df):,}')

# Label encode categorical features
_encoders = {}
_cat_cols = _ml_df.select_dtypes(include='object').columns.tolist()
for _col in _cat_cols:
    _le = LabelEncoder()
    _ml_df[_col] = _le.fit_transform(_ml_df[_col].astype(str))
    _encoders[_col] = _le
    print(f'  Encoded {_col}: {len(_le.classes_)} categories')

_X = _ml_df.drop('salary_usd', axis=1)
_y = _ml_df['salary_usd']
_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

print(f'\nTraining set: {len(_X_train):,} samples')
print(f'Test set: {len(_X_test):,} samples')
print(f'Features: {list(_X.columns)}')

# Train models
_t0_ml = _time6.time()

# Random Forest
_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
_rf.fit(_X_train, _y_train)
_rf_pred = _rf.predict(_X_test)
_rf_mae = mean_absolute_error(_y_test, _rf_pred)
_rf_r2 = r2_score(_y_test, _rf_pred)

# Gradient Boosting
_gb = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
_gb.fit(_X_train, _y_train)
_gb_pred = _gb.predict(_X_test)
_gb_mae = mean_absolute_error(_y_test, _gb_pred)
_gb_r2 = r2_score(_y_test, _gb_pred)

_ml_elapsed = _time6.time() - _t0_ml

print(f'\nModel Results (training time: {_ml_elapsed:.1f}s):')
print(f'  Random Forest:      MAE=${_rf_mae:,.0f}, R²={_rf_r2:.3f}')
print(f'  Gradient Boosting:  MAE=${_gb_mae:,.0f}, R²={_gb_r2:.3f}')

# Feature importance
_importances = _pd6.DataFrame({
    'feature': _X.columns,
    'rf_importance': _rf.feature_importances_,
    'gb_importance': _gb.feature_importances_
}).sort_values('gb_importance', ascending=False)
print(f'\nFeature importance (Gradient Boosting):')
for _idx in range(len(_importances)):
    _row = _importances.iloc[_idx]
    print(f'  {_row["feature"]:20s} RF={_row["rf_importance"]:.3f}  GB={_row["gb_importance"]:.3f}')

In [ ]:
# ── Cell 9: Cross-validation with hyperparameter search ──

_param_grid = [
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05},
    {'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1},
]

_t0_cv = _time6.time()
_cv_results = []

for _idx in range(len(_param_grid)):
    _params = _param_grid[_idx]
    _model = GradientBoostingRegressor(**_params, random_state=42)
    _scores = cross_val_score(_model, _X_train, _y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    _mae = -_scores.mean()
    _std = _scores.std()
    _cv_results.append((_params, _mae, _std))
    print(f'Config {_idx+1}: n={_params["n_estimators"]}, depth={_params["max_depth"]}, '
          f'lr={_params["learning_rate"]} → MAE=${_mae:,.0f} ± ${_std:,.0f}')

_cv_elapsed = _time6.time() - _t0_cv
print(f'\nCross-validation complete in {_cv_elapsed:.1f}s')

# Best config
_best = min(_cv_results, key=lambda x: x[1])
print(f'\nBest config: {_best[0]}')
print(f'Best MAE: ${_best[1]:,.0f} ± ${_best[2]:,.0f}')

In [ ]:
# ── Cell 10: Developer satisfaction & work patterns ──

# Find satisfaction-related columns
_satisfaction_cols = [c for c in _all_surveys.columns if 'satisf' in c.lower() or 'happy' in c.lower()]
print(f'Satisfaction columns: {_satisfaction_cols}')

# Work arrangement (remote/hybrid/in-person)
_remote_cols = [c for c in _all_surveys.columns if 'remote' in c.lower() or 'work' in c.lower()]
print(f'Work-related columns: {_remote_cols[:10]}')

# Remote work preference by year
_remote_col = None
for _col in _remote_cols:
    if 'remotework' in _col.lower() or 'remote' in _col.lower():
        _unique = _all_surveys[_col].dropna().unique()
        if len(_unique) < 10:  # Categorical
            _remote_col = _col
            break

if _remote_col:
    print(f'\nRemote work preference ({_remote_col}):')
    for _yr in _survey_years:
        _yr_data = _all_surveys[_all_surveys['survey_year'] == _yr]
        _remote_dist = _yr_data[_remote_col].value_counts(normalize=True)
        print(f'\n  {_yr}:')
        for _idx in range(min(5, len(_remote_dist))):
            _opt = _remote_dist.index[_idx]
            _pct = _remote_dist.iloc[_idx] * 100
            print(f'    {str(_opt):40s} {_pct:.1f}%')

# Education level distribution
_ed_col = None
for _col in _all_surveys.columns:
    if 'edlevel' in _col.lower():
        _ed_col = _col
        break

if _ed_col:
    print(f'\nEducation distribution ({_ed_col}):')
    _ed_dist = _all_surveys[_ed_col].value_counts(normalize=True)
    for _idx in range(min(8, len(_ed_dist))):
        _opt = _ed_dist.index[_idx]
        _pct = _ed_dist.iloc[_idx] * 100
        print(f'  {str(_opt):60s} {_pct:.1f}%')

# Age distribution
_age_col = None
for _col in _all_surveys.columns:
    if _col.lower() == 'age':
        _age_col = _col
        break

if _age_col:
    print(f'\nAge distribution:')
    _age_dist = _all_surveys[_age_col].value_counts(normalize=True).head(8)
    for _idx in range(len(_age_dist)):
        _opt = _age_dist.index[_idx]
        _pct = _age_dist.iloc[_idx] * 100
        print(f'  {str(_opt):40s} {_pct:.1f}%')

In [ ]:
# ── Cell 11: AI/ML tool adoption ──

# Find AI-related columns
_ai_cols = [c for c in _all_surveys.columns if 'ai' in c.lower() or 'copilot' in c.lower() or 'chatgpt' in c.lower() or 'llm' in c.lower()]
print(f'AI-related columns ({len(_ai_cols)}): {_ai_cols[:15]}')

# AI tool usage trends
_ai_tool_cols = [c for c in _all_surveys.columns if ('aitool' in c.lower() or 'aidev' in c.lower() or 'aisearch' in c.lower() or 'aiselect' in c.lower()) and ('have' in c.lower() or 'currently' in c.lower() or 'using' in c.lower() or 'select' in c.lower())]
if not _ai_tool_cols:
    _ai_tool_cols = [c for c in _ai_cols if 'tool' in c.lower() or 'select' in c.lower() or 'search' in c.lower()][:5]
print(f'AI tool columns: {_ai_tool_cols}')

for _col in _ai_tool_cols[:3]:
    print(f'\n{_col}:')
    for _yr in _survey_years:
        _yr_data = _all_surveys[_all_surveys['survey_year'] == _yr]
        _valid = _yr_data[_yr_data[_col].notna()]
        if len(_valid) == 0:
            continue
        # Count individual tools (semicolon separated)
        _tool_counter = Counter()
        for _val in _valid[_col].values:
            if isinstance(_val, str):
                for _tool in _val.split(';'):
                    _tool_counter[_tool.strip()] += 1
        if _tool_counter:
            print(f'  {_yr} (n={len(_valid):,}):')
            for _t, _c in _tool_counter.most_common(5):
                print(f'    {_t:30s} {100*_c/len(_valid):.1f}%')

# AI sentiment
_ai_sent_cols = [c for c in _all_surveys.columns if ('aisent' in c.lower() or 'aitrustworthy' in c.lower() or 'aiben' in c.lower() or 'aithreat' in c.lower())]
if _ai_sent_cols:
    print(f'\nAI sentiment columns: {_ai_sent_cols[:5]}')
    for _col in _ai_sent_cols[:2]:
        print(f'\n{_col}:')
        _dist = _all_surveys[_col].value_counts(normalize=True).head(6)
        for _idx in range(len(_dist)):
            print(f'  {str(_dist.index[_idx]):40s} {_dist.iloc[_idx]*100:.1f}%')

In [ ]:
# ── Cell 12: Summary statistics ──
print('=' * 60)
print('PROJECT 6: Stack Overflow Developer Survey - Summary')
print('=' * 60)
print(f'\nDataset: Stack Overflow Developer Survey {min(_survey_years)}-{max(_survey_years)}')
print(f'Total respondents: {len(_all_surveys):,}')
print(f'Years: {_survey_years}')
print(f'Columns: {len(_all_surveys.columns)}')
print(f'Memory: {_all_surveys.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'\nSalary Analysis:')
print(f'  Respondents with valid salary: {len(_with_salary):,}')
print(f'  Overall median salary: ${_with_salary["salary_usd"].median():,.0f}')
print(f'\nML Model Performance:')
print(f'  Random Forest:      MAE=${_rf_mae:,.0f}, R²={_rf_r2:.3f}')
print(f'  Gradient Boosting:  MAE=${_gb_mae:,.0f}, R²={_gb_r2:.3f}')
print(f'  Best CV config:     {_best[0]}')
print(f'  Best CV MAE:        ${_best[1]:,.0f}')
print(f'  ML training time:   {_ml_elapsed:.1f}s')
print(f'  CV search time:     {_cv_elapsed:.1f}s')
print('=' * 60)